In [11]:
import pandas as pd
import requests

API_URL = "https://archive-api.open-meteo.com/v1/archive"

LAT = 28.6519
LON = 77.2315
TIMEOUT = 10

START_DATE = "2026-04-01"   # ek din pehle, taaki 23:00->00:00 ke beech 23:05-23:55 generate ho
END_DATE   = "2026-06-11"

OUT_5M = r"C:\Users\suhan\Desktop\Final Year Project\PowerDemand\DataSet_Training\Delhi_Weather_5M.csv"

HOURLY_PARAMS = ",".join([
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "cloud_cover",
    "cloud_cover_low",
    "cloud_cover_mid",
    "cloud_cover_high",
])

params = {
    "latitude": LAT,
    "longitude": LON,
    "hourly": HOURLY_PARAMS,
    "timezone": "Asia/Kolkata",
    "start_date": START_DATE,
    "end_date": END_DATE,
}

r = requests.get(API_URL, params=params, timeout=TIMEOUT)
r.raise_for_status()
js = r.json()["hourly"]

hourly = pd.DataFrame({
    "datetime": pd.to_datetime(js["time"]),
    "Temperature (°C)": js["temperature_2m"],
    "Apparent Temperature (°C)": js["apparent_temperature"],
    "Relative Humidity (%)": js["relative_humidity_2m"],
    "Wind Speed (m/s)": js["wind_speed_10m"],
    "Precipitation (mm)": js["precipitation"],
    "Cloud Cover Total (%)": js["cloud_cover"],
    "Cloud Cover Low (%)": js["cloud_cover_low"],
    "Cloud Cover Mid (%)": js["cloud_cover_mid"],
    "Cloud Cover High (%)": js["cloud_cover_high"],
})

# 5-minute interpolation
five = hourly.set_index("datetime").sort_index()
full_idx = pd.date_range(five.index.min(), five.index.max(), freq="5min")
five = five.reindex(full_idx).interpolate(method="time", limit_direction="both")
five.index.name = "datetime"
five = five.reset_index()

five["Date"] = five["datetime"].dt.strftime("%d/%m/%Y")
five["TimeSlot"] = five["datetime"].dt.strftime("%H:%M")

five_out = five[
    ["Date","TimeSlot",
     "Temperature (°C)","Relative Humidity (%)","Apparent Temperature (°C)",
     "Precipitation (mm)","Wind Speed (m/s)",
     "Cloud Cover Total (%)","Cloud Cover Low (%)","Cloud Cover Mid (%)","Cloud Cover High (%)"]
]

# Purani file mein 01/04/2026 00:00-23:00 already hai -> skip karo, sirf 23:05+ rakho
mask_old_day = (five_out["Date"]=="01/04/2026") & (five_out["TimeSlot"]<="23:00")
five_out = five_out[~mask_old_day]

print("First new row:", five_out.iloc[0]["Date"], five_out.iloc[0]["TimeSlot"])
print("Last new row:", five_out.iloc[-1]["Date"], five_out.iloc[-1]["TimeSlot"])
print("Total new rows:", len(five_out))

five_out.to_csv(OUT_5M, mode="a", index=False, header=False)
print("Appended successfully!")

First new row: 01/04/2026 23:05
Last new row: 11/06/2026 23:00
Total new rows: 20448
Appended successfully!
